In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("dataset/AnnAttDataset.csv")

d = dataset.groupby("postId")["offensiveYN"].apply(list).to_dict()


# dataset info

In [3]:
count_tied = 0
count_onelabel = 0
for k,v in d.items():
    if len(set(v)) == 2: 
        count_tied+=1
    if len(set(v)) == 1: 
        count_onelabel+=1

print("Number of times more than one label was assigned to the text: ", count_tied)
print("Number of times one label was assigned to the text: ", count_onelabel)

Number of times more than one label was assigned to the text:  283
Number of times one label was assigned to the text:  344


In [4]:
df_language = dataset[["postId", "isAAE", "targetsBlackPeople", "vulgar"]]

df_language = df_language.drop_duplicates()
print(len(df_language))

627


In [5]:
count_all_false = 0
count_all_true = 0
for i, row in df_language.iterrows():
    aae = row["isAAE"]
    against_black = row["targetsBlackPeople"]
    vulgar = row["vulgar"]

    if aae == False and against_black == False and vulgar == False: 
        count_all_false +=1
    elif aae == True and against_black == True and vulgar == True: 
        count_all_true +=1

print("Number of times the text is neither in AAE, nor targets black people, nor contains vulgar language: ", count_all_false)
print("Number of times the text is in AAE, targets black people and contains vulgar language: ", count_all_true)

Number of times the text is neither in AAE, nor targets black people, nor contains vulgar language:  79
Number of times the text is in AAE, targets black people and contains vulgar language:  49


In [6]:
count_aae_vulgar = 0
count_aae_novulgar = 0
count_noaae_vulgar = 0
for i, row in df_language.iterrows():
    aae = row["isAAE"]
    against_black = row["targetsBlackPeople"]
    vulgar = row["vulgar"]

    if aae == True and vulgar == True: 
        count_aae_vulgar +=1
    elif aae == True and vulgar == False: 
        count_aae_novulgar +=1
    elif aae == False and vulgar == True: 
        count_noaae_vulgar +=1


print("both AAE and vulgar:", count_aae_vulgar)
print("AAE but NOT vulgar: ", count_aae_novulgar)
print("vulgar but NOT AAE: ", count_noaae_vulgar)

both AAE and vulgar: 190
AAE but NOT vulgar:  136
vulgar but NOT AAE:  173


In [12]:
print(df_language["isAAE"].value_counts())
print("-"*100, "\n")

print(df_language["targetsBlackPeople"].value_counts())
print("-"*100, "\n")

print(df_language["vulgar"].value_counts())
print("-"*100)

isAAE
True     326
False    301
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

targetsBlackPeople
False    458
True     169
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

vulgar
True     363
False    264
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------


# False negatives

annotators said it was offensive, and the model said it is not offensive

In [7]:
df_orig = pd.read_csv("qualitative_analysis/false_negative_llama_race_politics.csv")
print(df_orig.shape)
df = df_orig.drop_duplicates(subset="postId")
print(df.shape)

(254, 11)
(146, 11)


In [8]:
print("number of texts: ", len(df))

number of texts:  146


In [9]:
print(df["isAAE"].value_counts())
print("-"*100, "\n")

print(df["targetsBlackPeople"].value_counts())
print("-"*100, "\n")

print(df["vulgar"].value_counts())
print("-"*100)


isAAE
True     79
False    67
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

targetsBlackPeople
False    131
True      15
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

vulgar
False    75
True     71
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------


In [10]:
list_FN = df["postId"].tolist()

d_filtered = {}

for k,v in d.items():
    for i in list_FN: 
        if i==k: 
            d_filtered[k]=v

print(len(d_filtered))


146


In [11]:
count_tied = 0
count_onelabel = 0
to_read = []
for k,v in d_filtered.items():
    if len(set(v)) == 2: 
        count_tied+=1
    if len(set(v)) == 1: 
        count_onelabel+=1
        print(set(v))
        to_read.append(k)
        

print("Number of times more than one label was assigned to the text: ", count_tied)
print("Number of times one label was assigned to the text: ", count_onelabel)


{1}
{1}
{1}
{1}
{1}
{1}
{1}
{1}
Number of times more than one label was assigned to the text:  138
Number of times one label was assigned to the text:  8


In [12]:
df_toread = df[df["postId"].isin(to_read)]
print(df_toread.shape)

df_toread.to_csv("qualitative_analysis/false_neg_toread.csv", index=False)

(8, 11)


# False positives

annotators said it was not offensive, and the model said it is offensive

In [13]:
df = pd.read_csv("qualitative_analysis/false_positive_llama_race_politics.csv")
print(df.shape)
df = df.drop_duplicates(subset="postId")
print(df.shape)


(395, 11)
(168, 11)


In [14]:
print("number of texts: ", len(df))

number of texts:  168


In [15]:
print(df["isAAE"].value_counts())
print("-"*100, "\n")

print(df["targetsBlackPeople"].value_counts())
print("-"*100, "\n")

print(df["vulgar"].value_counts())
print("-"*100)


isAAE
True     101
False     67
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

targetsBlackPeople
False    133
True      35
Name: count, dtype: int64
---------------------------------------------------------------------------------------------------- 

vulgar
True     110
False     58
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------


In [16]:
list_FP = df["postId"].tolist()

d_filtered = {}

for k,v in d.items():
    for i in list_FP: 
        if i==k: 
            d_filtered[k]=v

print(len(d_filtered))

168


In [17]:
count_tied = 0
count_onelabel = 0
to_read = []
for k,v in d_filtered.items():
    if len(set(v)) == 2: 
        count_tied+=1
    if len(set(v)) == 1: 
        count_onelabel+=1
        print(set(v))
        to_read.append(k)

print("Number of times more than one label was assigned to the text: ", count_tied)
print("Number of times one label was assigned to the text: ", count_onelabel)

{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
{0}
Number of times more than one label was assigned to the text:  150
Number of times one label was assigned to the text:  18


In [18]:
df_toread = df[df["postId"].isin(to_read)]
print(df_toread.shape)

df_toread.to_csv("qualitative_analysis/false_pos_toread.csv", index=False)

(18, 11)


# The baseline is correct and race-politics is not

In [19]:
baseline = pd.read_csv("results/predictions/cleaned/cleaned_predictions_Llama_noCoT_baseline.csv")
race_political = pd.read_csv("results/predictions_dem/Llama/cleaned/cleaned_predictions_Llama_noCoT_race_political.csv")

baseline = baseline.rename(columns={"prediction": "baseline", "parsed_output": "expl_baseline"})
race_political = race_political.rename(columns={"prediction": "race_political", "parsed_output": "expl_race_political"})

baseline = baseline[["offensiveYN", "postId","annId", "baseline", "expl_baseline"]]
df = baseline.merge(race_political[["postId","annId", "race_political", "expl_race_political", "demographics"]], on=["postId", "annId"])

df = df.astype({'offensiveYN': 'int', 'baseline': 'int', "race_political": "int"})

In [20]:
df

,offensiveYN,postId,annId,baseline,expl_baseline,race_political,expl_race_political,demographics
0,0,545,10638687,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'liberal'}"
1,0,545,8526465,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'neutral'}"
2,0,3368,49021087,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'liberal'}"
3,0,3368,10166032,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'neutral'}"
4,1,1043,10638687,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it convey...,"{'race': 'black', 'political': 'liberal'}"
...,...,...,...,...,...,...,...,...
3067,1,1726,1226439,0,[the sentence is not offensive][because][it is...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}"
3068,1,751,27189963,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}"
3069,1,751,95971160,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}"
3070,1,305,15040445,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'white', 'political': 'liberal'}"


In [21]:
list_winner = []

for i,row in df.iterrows(): 
    baseline = row["baseline"]
    race_political = row["race_political"]
    gold = row["offensiveYN"]

    if gold == baseline and gold == race_political:
        list_winner.append("both")
    elif gold == baseline and gold != race_political: 
        list_winner.append("baseline")
    elif gold == race_political and gold != baseline: 
        list_winner.append("race_political")
    else: 
        list_winner.append("neither")

df["winner"]  =list_winner

In [22]:
df["winner"].value_counts()

winner
both              2351
neither            601
race_political      72
baseline            48
Name: count, dtype: int64

In [23]:
df

,offensiveYN,postId,annId,baseline,expl_baseline,race_political,expl_race_political,demographics,winner
0,0,545,10638687,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'liberal'}",both
1,0,545,8526465,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'neutral'}",both
2,0,3368,49021087,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'liberal'}",both
3,0,3368,10166032,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'black', 'political': 'neutral'}",both
4,1,1043,10638687,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it convey...,"{'race': 'black', 'political': 'liberal'}",both
...,...,...,...,...,...,...,...,...,...
3067,1,1726,1226439,0,[the sentence is not offensive][because][it is...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}",race_political
3068,1,751,27189963,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}",both
3069,1,751,95971160,1,[the sentence is offensive][because][it contai...,1,[the sentence is offensive][because][it contai...,"{'race': 'white', 'political': 'liberal'}",both
3070,1,305,15040445,0,[the sentence is not offensive][because][it is...,0,[the sentence is not offensive][because][it is...,"{'race': 'white', 'political': 'liberal'}",neither


In [24]:
pd.crosstab(df["winner"], df["offensiveYN"])

offensiveYN,0,1
winner,,
baseline,10,38
both,1040,1311
neither,385,216
race_political,57,15


In [25]:
df.to_csv("qualitative_analysis/comparison_baseline_race_political.csv")

# Embedding analysis

In [26]:
import numpy as np
import os
from glob import glob
import pandas as pd

In [27]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/samuele/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [28]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from glob import glob
import os
import matplotlib.pyplot as plt


folder = "results/output_expl_analysis/positive"
dataframes_dir = f"{folder}/dataframes"
pattern = "clusters_*.csv"
for file in glob(os.path.join(dataframes_dir, pattern)):
    len_sent = []
    len_word = []
    print(file)
    df = pd.read_csv(file)
    print(df.shape)
    for i,row in df.iterrows(): 
        text = row["reasoning"]
        sent = sent_tokenize(text)
        words = word_tokenize(text)

        len_sent.append(len(sent))
        len_word.append(len(words))

    df["len_sent"] = len_sent
    df["len_word"] = len_word

    print(df.groupby("cluster")["len_word"].mean())
    print(df.groupby("cluster")["len_word"].std())
    print("-"*100)
    print()

    fig_filename = file.split("/")[-1].replace(".csv", "")
    for cluster in df['cluster'].unique():
        subset = df[df['cluster'] == cluster]['len_word']
        subset.plot.kde(label=f'Cluster {cluster}')  # KDE gives smooth density curve

    plt.xlabel("Word Length")
    plt.ylabel("Density")
    plt.title(f"{fig_filename}")
    plt.legend()
    plt.savefig(f"{folder}/kde_plot/{fig_filename}.png")
    plt.close()

results/output_expl_analysis/positive/dataframes/clusters_political.csv
(933, 23)
cluster
0    11.566553
1    10.141210
Name: len_word, dtype: float64
cluster
0    4.420988
1    4.417543
Name: len_word, dtype: float64
----------------------------------------------------------------------------------------------------

results/output_expl_analysis/positive/dataframes/clusters_gender_race_political.csv
(1059, 23)
cluster
0    10.329356
1    16.011321
2     9.704000
Name: len_word, dtype: float64
cluster
0    3.391006
1    5.064346
2    3.846710
Name: len_word, dtype: float64
----------------------------------------------------------------------------------------------------

results/output_expl_analysis/positive/dataframes/clusters_gender_race.csv
(1029, 23)
cluster
0     6.208333
1    10.217270
2    16.233645
3    10.250000
Name: len_word, dtype: float64
cluster
0    2.358687
1    3.735326
2    5.462771
3    3.182391
Name: len_word, dtype: float64
---------------------------------------

In [ ]:
folder = "results/output_expl_analysis/negative"
dataframes_dir = f"{folder}/dataframes"
pattern = "clusters_*.csv"
for file in glob(os.path.join(dataframes_dir, pattern)):
    len_sent = []
    len_word = []
    print(file)
    df = pd.read_csv(file)
    print(df.shape)
    for i,row in df.iterrows(): 
        text = row["reasoning"]
        sent = sent_tokenize(text)
        words = word_tokenize(text)

        len_sent.append(len(sent))
        len_word.append(len(words))

    df["len_sent"] = len_sent
    df["len_word"] = len_word

    print(df.groupby("cluster")["len_word"].mean())
    print(df.groupby("cluster")["len_word"].std())
    print("-"*100)
    print()


    fig_filename = file.split("/")[-1].replace(".csv", "")
    for cluster in df['cluster'].unique():
        subset = df[df['cluster'] == cluster]['len_word']
        subset.plot.kde(label=f'Cluster {cluster}')  # KDE gives smooth density curve

    plt.xlabel("Word Length")
    plt.ylabel("Density")
    plt.title(f"{fig_filename}")
    plt.legend()
    plt.savefig(f"{folder}/kde_plot/{fig_filename}.png")
    plt.close()

embeddings/negative/clusters_political.csv
(616, 23)
cluster
0    17.275862
1    10.212314
Name: len_word, dtype: float64
cluster
0    4.786385
1    3.189333
Name: len_word, dtype: float64
----------------------------------------------------------------------------------------------------

embeddings/negative/clusters_gender_race.csv
(711, 23)
cluster
0    15.378788
1     9.979866
Name: len_word, dtype: float64
cluster
0    5.113957
1    3.007024
Name: len_word, dtype: float64
----------------------------------------------------------------------------------------------------

embeddings/negative/clusters_race_political.csv
(738, 23)
cluster
0    10.060120
1    15.887029
Name: len_word, dtype: float64
cluster
0    3.007419
1    4.195811
Name: len_word, dtype: float64
----------------------------------------------------------------------------------------------------

embeddings/negative/clusters_gender.csv
(641, 23)
cluster
0     9.862233
1    15.768182
Name: len_word, dtype: float64
c